In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
import plotly.express as px
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_fields

In [ ]:
day = 7
ρs = [0.0,0.2,0.3, 0.4,0.5,0.6,0.7,0.8,0.9,0.99]
demand= []
random_demand = []
reserve = []
energy_reserve = []
for ρ in ρs: 
    i = f'base_case_increased_storage_energy_v8.{ρ}.4'
    demand_ = pd.read_csv(os.path.join("..", "input", i, 'uc','Demand.csv'))
    random_demand_ = pd.read_csv(os.path.join("..", "input", i, 'ed','random_demand.csv'))
    reserve_ = pd.read_csv(os.path.join("..", "input", i, 'uc','Reserve.csv'))
    energy_reserve_ = pd.read_csv(os.path.join("..", "input", i, 'uc','Energy reserve.csv')) 
    add_fields(demand_, ρ=ρ)
    add_fields(random_demand_, ρ=ρ)
    add_fields(reserve_, ρ=ρ)
    add_fields(energy_reserve_, ρ=ρ)
    demand.append(demand_)
    random_demand.append(random_demand_)
    reserve.append(reserve_)
    energy_reserve.append(energy_reserve_)

demand = pd.concat(demand).set_index(['day','ρ', 'hour'])
random_demand = pd.concat(random_demand).set_index(['day','ρ', 'hour'])
reserve = pd.concat(reserve).set_index(['day','ρ', 'hour'])
energy_reserve = pd.concat(energy_reserve).set_index(['day','ρ', 'i_hour','t_hour'])





In [ ]:
reserve = reserve.loc[day]   
reserve.reserve_down_MW = -1*reserve.reserve_down_MW

energy_reserve = energy_reserve.loc[day]   
energy_reserve.reserve_down_MW = -1*energy_reserve.reserve_down_MW

imbalance = random_demand.sub(demand['demand'], axis=0, level=['day','ρ','hour',])
imbalance = imbalance.loc[day,:]
# cum_imbalance = imbalance.groupby(['ρ']).cumsum()

In [ ]:
tuples = [(r, h_i, h) for r, h in imbalance.index for h_i in imbalance.index.get_level_values('hour').unique() if h_i <= h]
cum_imbalance = pd.DataFrame(tuples, columns=['ρ', 'i_hour', 't_hour'])   
cum_imbalance = cum_imbalance.merge(imbalance, left_on=['ρ','t_hour'], right_on=['ρ','hour']) 
cum_imbalance.set_index(['ρ','i_hour','t_hour'], inplace=True)
cum_imbalance = cum_imbalance.groupby(['ρ','i_hour']).cumsum()

In [ ]:
# empirical_mu_sf = "20250401 - empirical_mu.csv"
# empirical_mu_sf = "02042025 - empirical_mu.csv"
empirical_mu_sf = "20250402 - water_tank_empirical_mu_.csv"
empirical_mu = pd.read_csv(os.path.join("archives", "ESA",empirical_mu_sf)) 
empirical_mu.rename(columns={'rho': 'ρ'}, inplace=True)
empirical_mu = empirical_mu.melt(
    id_vars=['mu', 'ρ'], 
    var_name='hour', 
    value_name='value'
)
empirical_mu['hour'] = pd.to_numeric(empirical_mu['hour'], errors='coerce')
empirical_mu.set_index(['mu','ρ','hour',], inplace=True)
avg_empirical_mu = empirical_mu.groupby(['mu','ρ']).mean()
# imbalance.mul(empiral_mu.loc['θUPDIS/RESUPDIS',:], axis=1, level=['hour', 'ρ'])
mu_up = 'θUPDIS/RESUPDIS' if 'θUPDIS/RESUPDIS' in empirical_mu.index.get_level_values('mu') else 'θUP/RESUP'
mu_dn = 'θUPDIS/RESUPDIS' if 'θUPDIS/RESUPDIS' in empirical_mu.index.get_level_values('mu') else 'θDN/RESDN'    


In [ ]:
common_index = reserve.index.intersection(empirical_mu.index.droplevel('mu'))
weigthed_reserve = reserve.loc[common_index].copy()
weigthed_reserve['weighted_reserve_up_MW'] = weigthed_reserve.reserve_up_MW*empirical_mu.loc[mu_up].loc[common_index,'value']
weigthed_reserve['weighted_reserve_down_MW'] = weigthed_reserve.reserve_down_MW*empirical_mu.loc[mu_dn].loc[common_index,'value']

# .mul(empirical_mu.loc[mu_up].loc[common_index,'value'], axis=0)

# weigthed_reserve.rename(columns={"reserve_up_MW": "weighted_reserve_up_MW", "reserve_down_MW": "weighted_reserve_down_MW"}, inplace=True)
cum_weigthed_reserve = weigthed_reserve.groupby(['ρ']).cumsum()
cum_reserve = reserve.groupby(['ρ']).cumsum()


In [ ]:
tuples = [(r, h_i, h) for r, h in weigthed_reserve.index for h_i in weigthed_reserve.index.get_level_values('hour').unique() if h_i <= h]
cum_weigthed_reserve = pd.DataFrame(tuples, columns=['ρ', 'i_hour', 't_hour'])   
cum_weigthed_reserve = cum_weigthed_reserve.merge(weigthed_reserve, left_on=['ρ','t_hour'], right_on=['ρ','hour']) 
cum_weigthed_reserve.set_index(['ρ','i_hour','t_hour'], inplace=True)
cum_weigthed_reserve = cum_weigthed_reserve.groupby(['ρ','i_hour']).cumsum()

In [ ]:
merged_data = cum_weigthed_reserve.join(cum_imbalance, how='inner').stack()
merged_data = merged_data.reset_index()
merged_data.rename(columns={'level_3': 'type', 0: 'value'}, inplace=True)

merged_data = merged_data[merged_data['ρ'].isin([0.2, 0.3, 0.4, 0.5])]
merged_data = merged_data.loc[merged_data.i_hour == merged_data.i_hour.min()]
px.line(
    merged_data,
    x='t_hour',
    y='value',
    color='type',
    facet_col='ρ',
    labels={'value': 'MWh'},
    title='Cumulative of X_t, µ*R^up, -µ*R^dn'
)


In [ ]:
result_up =  cum_imbalance.sub(cum_weigthed_reserve['weighted_reserve_up_MW'], axis=0)
result_dn =  cum_imbalance.sub(cum_weigthed_reserve['weighted_reserve_down_MW'], axis=0)
# Reset the index if needed
# Calculate the probability of being lower than zero for each index
prob_imb_big_up = (result_up > 0).mean(axis=1)
prob_imb_loq_dn = (result_dn < 0).mean(axis=1)
# Combine the probabilities into a single DataFrame
probabilities = pd.concat([prob_imb_big_up, prob_imb_loq_dn], axis=1)
probabilities.columns = ['prob_imb_big_up', 'prob_imb_low_dn']


In [ ]:
energy_reserve['reserve_down_MW']

In [ ]:
result_up =  cum_imbalance.sub(energy_reserve['reserve_up_MW'], axis=0)
result_dn =  cum_imbalance.sub(energy_reserve['reserve_down_MW'], axis=0)
# Reset the index if needed
# Calculate the probability of being lower than zero for each index
prob_imb_big_up = (result_up > 0).mean(axis=1)
prob_imb_loq_dn = (result_dn < 0).mean(axis=1)
# Combine the probabilities into a single DataFrame
probabilities_energy = pd.concat([prob_imb_big_up, prob_imb_loq_dn], axis=1)
probabilities_energy.columns = ['prob_imb_big_up', 'prob_imb_low_dn']

In [ ]:

to_plot = probabilities.stack().reset_index().rename(columns={'level_3': 'type', 0: 'prob'})
to_plot['i_hour'] = to_plot['i_hour'] - to_plot['i_hour'].min() + 1
to_plot['t_hour'] = to_plot['t_hour'] - to_plot['t_hour'].min() + 1
fig = px.scatter(
    to_plot,
    y='t_hour',
    x='i_hour',
    color='prob',
    facet_col='ρ',
    facet_row='type',
    color_continuous_scale='Viridis',
    labels={'prob': 'Probability'},
    title='P_{i,t}^up (upper graphs) and P_{i,t}^dn (lower graphs)',
)
fig.show()


In [ ]:
to_plot = probabilities_energy.stack().reset_index().rename(columns={'level_3': 'type', 0: 'prob'})
fig = px.scatter(
    to_plot,
    y='t_hour',
    x='i_hour',
    color='prob',
    facet_col='ρ',
    facet_row='type',
    color_continuous_scale='Viridis',
    labels={'prob': 'Probability'},
    title='Scatter Plot of Probabilities'
)
fig.show()

In [ ]:
to_plot = probabilities.stack().reset_index().rename(columns={'level_3': 'type', 0: 'prob'})
to_plot = to_plot.loc[to_plot.i_hour == to_plot.i_hour.min()]
fig = px.scatter(to_plot, x='t_hour', y='prob', facet_col='ρ', facet_row='type', title='P^up{1,t} and P^dn{1,t}')
fig.add_traces(
    list(px.line(to_plot, x='t_hour', y='prob', facet_col='ρ', facet_row='type').select_traces())
)
fig.show()


In [ ]:
to_plot = probabilities_energy.stack().reset_index().rename(columns={'level_3': 'type', 0: 'prob'})
to_plot = to_plot.loc[to_plot.i_hour == to_plot.i_hour.min()+15]
fig = px.scatter(to_plot, x='t_hour', y='prob', facet_col='ρ', facet_row='type', title='Scatter and Line Plot')
fig.add_traces(
    list(px.line(to_plot, x='t_hour', y='prob', facet_col='ρ', facet_row='type').select_traces())
)
fig.show()

In [ ]:
cum_weigthed_reserve

In [ ]:

# # Merge cum_weigthed_reserve and cum_imbalance on their common indices
# cum_weigthed_reserve_aux = cum_weigthed_reserve.copy()
# cum_weigthed_reserve_aux.weighted_reserve_down_MW = -cum_weigthed_reserve_aux.weighted_reserve_down_MW
# merged_data = cum_weigthed_reserve_aux.join(cum_imbalance, how='inner').stack()

# # Reset the index for plotting
# to_plot = merged_data.reset_index()

# # Negate reserve_down_MW for plotting
# # to_plot.weighted_reserve_down_MW = -to_plot.weighted_reserve_down_MW

# # Extract all demand columns from cum_imbalance
# demand_columns = cum_imbalance.columns.tolist()

# # Create the plot
# fig = px.line(
#     to_plot, 
#     x='hour', 
#     y=['reserve_up_MW', 'reserve_down_MW'] + demand_columns, 
#     color='ρ', 
#     labels={'value': 'Reserve (MW)', 'variable': 'Type'}, 
#     title='Reserve Up, Down, and Demand by Hour'
# )

# # Update layout
# fig.update_layout(xaxis_title='Hour', yaxis_title='Reserve and Demand (MW)')

# # Show the plot
# fig.show()


In [ ]:
to_plot

In [ ]:
weigthed_reserve

In [ ]:
imbalance

In [ ]:
reserve

In [ ]:
# reserve_day_7 = reserve[reserve['day'] == 7]
fig = px.line(reserve, x='hour', y='reserve_up_MW', color='ρ')
fig.show()

In [ ]:
# Filter the dataframe for day = 7
reserve_day_7 = reserve[reserve['day'] == 7]

# Plot the boxplot
fig = px.box(reserve_day_7, x='hour', y='reserve_up_MW')
fig.update_layout(xaxis_title='Hour', yaxis_title='Reserve Up (MW)', title='Boxplot of Reserve Up (MW) for Day 7')
fig.show()


In [ ]:
filter = (reserve['day'] == 7)# & (reserve['hour'] == 150)
reserve_day_7 = reserve[filter]

# Plot the boxplot
fig = px.box(reserve_day_7, x='ρ', y='reserve_up_MW')
fig.update_layout(xaxis_title='ρ', yaxis_title='Reserve Up (MW)', title='Boxplot of Reserve Up (MW) for Day 7')
fig.show()